In [ ]:
#pip install joblib
import os
import multiprocessing as mp
from joblib import Parallel, delayed
from hdf5_creation import prepare_for_hdf5, update_hdf5

In [ ]:
def fragment_join(subject, night, type):
    base = f"D:/EEG_Data_stage/{subject}/iEEG/{type}"
    score_base = f"D:/EEG_Data_stage/{subject}/iEEG/U_sleep_API_10s"
    files = []
    scores = []
    for file in os.listdir(base):
        if f"night{night}" in file and ".vhdr" in file:
            files.append(os.path.join(base, file))
            scores.append(os.path.join(score_base, file.replace(".vhdr", "_hypnogram.npy")))
    return files, scores

In [ ]:
fs = 250 # EEG sampling frequency
epoch_length = 10 #in seconds

base_dir = "D:/EEG_Data_stage/features_intra"

hdf5_path = 'D:/EEG_Data_stage/datasets/intra_cranial_dataset.h5'   #Name of the new hdf5 file to create

files = os.listdir(base_dir)

num_processes = mp.cpu_count()
print('Number of processes :', num_processes)

results = (
    Parallel(
        n_jobs=min(num_processes, len(files)), verbose = 10)
           (
        delayed(
               prepare_for_hdf5)(recording, fs, base_dir, epoch_length) for recording in files))

for result in results:
    print(result[2])
    update_hdf5(result, hdf5_path)
